# Fine-tune `Qwen/Qwen3-VL-8B-Instruct` with QLoRA on Amazon SageMaker AI using ModelTrainer

In this notebook, we will fine-tune a **vision-language model** on Amazon SageMaker AI, using Python scripts and SageMaker ModelTrainer for executing a training job, then deploy it to a real-time endpoint with the vLLM Deep Learning Container.

Qwen3-VL is a multimodal checkpoint, so it needs a few things that a text-only model does not. This notebook ships its own `scripts_vl/` (not reusing `scripts/`):

| | Text-only (`02.01`) | Vision-language (this notebook) |
| --- | --- | --- |
| Auto class | `AutoModelForCausalLM` | `AutoModelForImageTextToText` |
| Adapter merge | `AutoPeftModelForCausalLM` | explicit `PeftModel.from_pretrained(base, adapter)` |
| Saved with the model | tokenizer | tokenizer **and** `AutoProcessor` |
| `transformers` | `4.52.2` | `>= 4.57` (Qwen3-VL support) |
| Precision | fp16 | bf16 (`flash_attention_2`) |
| Serving container | DJL-LMI | vLLM DLC (serves images out of the box) |

We fine-tune on a **text-only** medical reasoning dataset, so only the language model actually learns. `target_modules="all-linear"` does attach LoRA to the vision tower as well, but a text-only batch carries no `pixel_values`: the tower never runs, those adapters receive no gradient, and the merge writes them back unchanged. The served vision weights are therefore identical to the base checkpoint, and the endpoint still accepts images afterwards.

## Prerequisites

In [ ]:
%pip install -q datasets==3.2.0 "transformers>=4.57" pandas matplotlib pillow

## This cell will restart the kernel. Click "OK".

In [ ]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)

***

## Setup Configuration file path

If you have created a Managed MLflow server, copy the `ARN` code here and assign a name to the experiment

In [ ]:
import boto3
import shutil
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.config import load_sagemaker_config

sagemaker_session = Session()
s3_client = boto3.client('s3')

region = sagemaker_session.boto_session.region_name
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
configs = load_sagemaker_config()

print(f"Region: {region}")
print(f"Default S3 bucket: {bucket_name}")

If you have your own MLflow tracking server, update the `TrackingServerName` value below to enable experiment tracking.

In [ ]:
from botocore.exceptions import ClientError

try:
    response = boto3.client('sagemaker').describe_mlflow_tracking_server(
        TrackingServerName='genai-mlflow-tracker'
    )
    mlflow_tracking_server_uri = response['TrackingServerArn']
except ClientError:
    mlflow_tracking_server_uri = ""

if mlflow_tracking_server_uri == "":
    print("No MLflow Tracking Server Found, experiments will not be tracked.")
else:
    print(f"MLflow Tracking Server ARN: {mlflow_tracking_server_uri}")

In [ ]:
import os

os.environ["mlflow_uri"] = mlflow_tracking_server_uri
os.environ["mlflow_experiment_name"] = "Qwen3-VL-8B-Instruct-sft"

***

## Visualize and upload the dataset

We are going to load the [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset. It is text-only, so this run only trains the language model — the vision tower's adapters get no gradient and merge back unchanged.

In [ ]:
from datasets import load_dataset
import pandas as pd

num_samples = 100

full_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split=f"train[:{num_samples}]")

full_dataset[0]

In [ ]:
train_test_split_datasets = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split_datasets["train"]
test_dataset = train_test_split_datasets["test"]

print(f"Number of train elements: {len(train_dataset)}")
print(f"Number of test elements: {len(test_dataset)}")

Create a prompt template and convert each sample into the chat `messages` schema that `SFTTrainer` expects.

In [ ]:
SYSTEM_PROMPT = """You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response."""


# template dataset to add prompt to each sample
def convert_to_messages(sample, system_prompt=""):
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sample["Question"]},
        {"role": "assistant", "content": f"{sample['Complex_CoT']}\n\n{sample['Response']}"}
    ]

    sample["messages"] = messages
    
    return sample

In [ ]:
from random import randint

train_dataset = train_dataset.map(convert_to_messages, remove_columns=list(full_dataset.features), fn_kwargs={"system_prompt": SYSTEM_PROMPT})
test_dataset = test_dataset.map(convert_to_messages, remove_columns=list(full_dataset.features), fn_kwargs={"system_prompt": SYSTEM_PROMPT})

#grab a sample from the training and test sets
print(f"Train Sample:\n{train_dataset[randint(0, len(train_dataset)-1)]}\n\n")
print(f"Test Sample:\n{test_dataset[randint(0, len(test_dataset)-1)]}\n\n")

### Upload to Amazon S3

In [ ]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f'{default_prefix}/datasets/qwen3-vl-fine-tuning-modeltrainer-sft'
else:
    input_path = f'datasets/qwen3-vl-fine-tuning-modeltrainer-sft'

# Save datasets to s3
train_dataset.to_json("./data/train/dataset.json", orient="records")
test_dataset.to_json("./data/test/dataset.json", orient="records")

s3_client.upload_file("./data/train/dataset.json", bucket_name, f"{input_path}/train/dataset.json")
train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.json"
s3_client.upload_file("./data/test/dataset.json", bucket_name, f"{input_path}/test/dataset.json")
test_dataset_s3_path = f"s3://{bucket_name}/{input_path}/test/dataset.json"

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(test_dataset_s3_path)

In [ ]:
from utils import plot_length_distribution

plot_length_distribution(
    train_dataset=train_dataset,
    validation_dataset=test_dataset,
    bins=20,
    figsize=(10, 6)
)

***

## Model fine-tuning

We are now ready to fine-tune our model.

`use_local_model = False` lets the training job pull the weights from the Hugging Face Hub directly. For a 16 GB vision-language checkpoint that avoids staging the model through this space and your S3 bucket. Set it to `True` if you need a pinned copy in S3.

In [ ]:
model_id = "Qwen/Qwen3-VL-8B-Instruct"
model_id_filesafe = model_id.replace("/","_")

use_local_model = False #set to false for the training job to download from HF, otherwise True will download locally

In [ ]:
from huggingface_hub import snapshot_download
import os
import subprocess

if use_local_model:
    model_local_location = f"../models/{model_id_filesafe}"
    prefix = f"{default_prefix}/" if default_prefix else ""
    model_s3_destination = f"s3://{bucket_name}/{prefix}models/{model_id_filesafe}"
    
    print(f"Downloading model {model_id}")
    os.makedirs(model_local_location, exist_ok=True)
    snapshot_download(repo_id=model_id, local_dir=model_local_location)
    
    print("Beginning Model Upload...")
    subprocess.run(['aws', 's3', 'sync', model_local_location, model_s3_destination, 
                   '--exclude', '.cache/*', '--exclude', '.gitattributes'])
    
    print(f"Model uploaded to:\n{model_s3_destination}")
    os.environ["model_location"] = model_s3_destination
else:
    os.environ["model_location"] = model_id

Write the training config. Four values differ from the text-only notebook, in two groups, and both groups matter for Qwen3-VL:

- **`bf16: true` / `fp16: false`.** `train.py` only selects `flash_attention_2` on the bf16 branch, and the merge step loads the base model in bf16. Flipping these to fp16 changes the attention implementation and the merge dtype.
- **`per_device_train_batch_size: 1` with `gradient_accumulation_steps: 4`.** An 8B model with a vision tower needs the smaller per-device batch on a single 24 GB A10G; the accumulation steps keep the effective batch size at 4.

In [ ]:
%%bash

cat > ./args.yaml <<EOF

# MLflow Config
mlflow_uri: "${mlflow_uri}"
mlflow_experiment_name: "${mlflow_experiment_name}"


model_id: "${model_location}"       # Hugging Face model id, or S3 location

# sagemaker specific parameters
output_dir: "/opt/ml/model"                       # path to where SageMaker will upload the model 
train_dataset_path: "/opt/ml/input/data/train/"   # path to where FSx saves train dataset
test_dataset_path: "/opt/ml/input/data/test/"     # path to where FSx saves test dataset
# training parameters
max_seq_length: 1500
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1                 
learning_rate: 2e-4                    # learning rate scheduler
num_train_epochs: 1                    # number of training epochs
per_device_train_batch_size: 1         # batch size per device during training
per_device_eval_batch_size: 1          # batch size for evaluation
gradient_accumulation_steps: 4         # number of steps before performing a backward/update pass
gradient_checkpointing: true           # use gradient checkpointing
fp16: false
bf16: true                             # Qwen3-VL: bf16 selects the flash_attention_2 path
tf32: false

merge_weights: true                    # merge the QLoRA adapter into the base model
EOF

Lets upload the config file to S3.

In [ ]:
from sagemaker.core.s3 import S3Uploader

if default_prefix:
    input_path = f"s3://{bucket_name}/{default_prefix}/training_config/{model_id_filesafe}"
else:
    input_path = f"s3://{bucket_name}/training_config/{model_id_filesafe}"

# upload the model yaml file to s3
model_yaml = "args.yaml"
train_config_s3_path = S3Uploader.upload(local_path=model_yaml, desired_s3_uri=f"{input_path}/config")

print(f"Training config uploaded to:")
print(train_config_s3_path)

## Fine-tune model

Below will train the model with QLoRA, merge the adapter in the base model and save in S3.

The training entry point is `./scripts_vl/train.py`. `type(config) in AutoModelForImageTextToText._model_mapping` check so text-only models still take the original path:

1. Loads through `AutoModelForImageTextToText` instead of `AutoModelForCausalLM`.
2. Does not pass `use_cache` to `from_pretrained` — Qwen3-VL's composite `__init__` rejects it — and sets `config.text_config.use_cache` after loading instead.
3. Merges the adapter with an explicit `PeftModel.from_pretrained(base, adapter_dir)`, because `AutoPeftModelForCausalLM` assumes a causal-LM architecture.
4. Saves the `AutoProcessor` next to the tokenizer, so the merged checkpoint keeps `preprocessor_config.json` and can be served as a multimodal model.

#### Get PyTorch image_uri

In [ ]:
instance_type = "ml.g5.2xlarge"

instance_type

In [ ]:
from sagemaker.core import image_uris

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.6.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

In [ ]:
from sagemaker.train.model_trainer import ModelTrainer, InputData, Torchrun, StoppingCondition
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.core.shapes import OutputDataConfig

# Define the script to be run — scripts_vl holds the vision-language variant of train.py
source_code = SourceCode(
    source_dir="./scripts_vl",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=100
)

# define Training Job Name 
job_name = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft-script"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=14400
    ),
    hyperparameters={
        "config": "/opt/ml/input/data/config/args.yaml"
    },
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    environment={"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"},
    role=get_execution_role(sagemaker_session, use_default=True),
)

In [ ]:
# Pass the input data
train_input = InputData(
    channel_name="train",
    data_source=train_dataset_s3_path,
)

test_input = InputData(
    channel_name="test",
    data_source=test_dataset_s3_path,
)

config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path,
)

# Check input channels configured
data = [train_input, test_input, config_input]
data

The reference run took **15 minutes 38 seconds** on a single `ml.g5.2xlarge` (100 samples, 1 epoch), including downloading the 16 GB checkpoint from the Hub and merging the adapter.


In [ ]:
# starting the train job with our uploaded datasets as input
model_trainer.train(input_data_config=data, wait=True)

***

# Model Deployment

In the following sections, we are going to deploy the fine-tuned model on an Amazon SageMaker Real-time endpoint.

## Load Fine-Tuned model

In [ ]:
import sys
from utils import get_last_job_name

job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft-script"

job_name = get_last_job_name(job_prefix)

job_name

#### Inference configurations

We serve the merged checkpoint with the **vLLM Deep Learning Container**. vLLM resolves Qwen3-VL as `Qwen3VLForConditionalGeneration`, loads the image processor from the merged checkpoint, and exposes an OpenAI-compatible chat API that accepts image content parts — no extra multimodal configuration required.

In [ ]:
instance_count = 1
instance_type = "ml.g7e.2xlarge"
health_check_timeout = 1800

In [ ]:
inference_image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.25.1-gpu-py312-cu130-ubuntu22.04-sagemaker"
print(f"using image to host: {inference_image_uri}")

`SM_VLLM_*` environment variables are translated into `vllm serve` flags by the container's entrypoint (`SM_VLLM_MAX_MODEL_LEN` becomes `--max-model-len`).

In [ ]:
import json
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

role = get_execution_role(sagemaker_session, use_default=True)

if default_prefix:
    model_data=f"s3://{bucket_name}/{default_prefix}/{job_prefix}/{job_name}/output/model.tar.gz"
else:
    model_data=f"s3://{bucket_name}/{job_prefix}/{job_name}/output/model.tar.gz"

deploy_model_name = f"Qwen3-VL-8B-sft-{job_name[-8:]}"

core_model = Model.create(
    model_name=deploy_model_name,
    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=inference_image_uri,
        model_data_url=model_data,
        environment={
            'HF_MODEL_ID': "/opt/ml/model",
            'SM_VLLM_MAX_MODEL_LEN': '8192',
            'SM_VLLM_TENSOR_PARALLEL_SIZE': '1',
            'SM_VLLM_GPU_MEMORY_UTILIZATION': '0.90',
        },
    ),
)

deploy_model_name

In [ ]:
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.helper.session_helper import _wait_until, _deploy_done

endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft"
TUNED_ENDPOINT_NAME = name_from_base(endpoint_name)

EndpointConfig.create(
    endpoint_config_name=TUNED_ENDPOINT_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=deploy_model_name,
            initial_instance_count=instance_count,
            instance_type=instance_type,
            container_startup_health_check_timeout_in_seconds=health_check_timeout,
            model_data_download_timeout_in_seconds=3600,
        )
    ],
)

core_endpoint = Endpoint.create(
    endpoint_name=TUNED_ENDPOINT_NAME,
    endpoint_config_name=TUNED_ENDPOINT_NAME,
)

_wait_until(lambda: _deploy_done(sagemaker_session.sagemaker_client, TUNED_ENDPOINT_NAME), poll=30)
core_endpoint = Endpoint.get(endpoint_name=TUNED_ENDPOINT_NAME)
print(f"Endpoint status: {core_endpoint.endpoint_status}")

#### Predict

In [ ]:
SYSTEM_PROMPT = f"""You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response."""

USER_PROMPT = "A 3-week-old child has been diagnosed with late onset perinatal meningitis, and the CSF culture shows gram-positive bacilli. What characteristic of this bacterium can specifically differentiate it from other bacterial agents?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

messages

vLLM speaks the OpenAI chat schema, so generation settings sit at the top level of the body (`max_tokens`, `temperature`) instead of inside the `parameters` dict the DJL-LMI container expects.

In [ ]:
payload = json.dumps({
	"messages": messages,
    "temperature": 0.2,
    "top_p": 0.9,
    "max_tokens": 1024
})

response = core_endpoint.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))
result["choices"][0]["message"]["content"]

#### Predict with an image

The fine-tuning data was text-only, but because `train.py` saved the `AutoProcessor` alongside the merged weights, the endpoint is still a working multimodal model. Images are passed as OpenAI-style content parts.

We build a small image locally and send it as a `data:` URL. Base64 keeps the request inside SageMaker's 6 MB invoke payload limit and avoids the container having to fetch a remote URL, which many hosts block.

In [ ]:
import base64
import io
from PIL import Image, ImageDraw

# a simple, unambiguous test image: a green triangle on a black background
image = Image.new("RGB", (256, 256), "black")
draw = ImageDraw.Draw(image)
draw.polygon([(128, 40), (30, 210), (226, 210)], fill="green")

buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

image

In [ ]:
payload = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in one short sentence."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            ],
        }
    ],
    "temperature": 0.0,
    "max_tokens": 64,
})

response = core_endpoint.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))
print(result["choices"][0]["message"]["content"])
print()
print(f"prompt tokens (text + image): {result['usage']['prompt_tokens']}")

If the reply describes a green triangle, the vision path is genuinely working. A model whose image input is silently dropped will still answer confidently — it just describes something that is not there — so this is worth checking rather than assuming.

### Store variables

Save the endpoint name for use later

In [ ]:
%store TUNED_ENDPOINT_NAME

## Clean up

A real-time endpoint bills for as long as it exists. Set `cleanup = True` to delete the endpoint, endpoint config and model created by this notebook.

In [ ]:
cleanup = False

if cleanup:
    core_endpoint.delete()
    EndpointConfig.get(endpoint_config_name=TUNED_ENDPOINT_NAME).delete()
    Model.get(model_name=deploy_model_name).delete()
    print(f"Deleted endpoint, endpoint config and model for {TUNED_ENDPOINT_NAME}")
else:
    print(f"Endpoint {TUNED_ENDPOINT_NAME} left running. Set cleanup = True to delete it.")